# Campus Waste Intelligence System
## Google Colab Notebook

This notebook implements the full project pipeline:

1. Data download hooks for the datasets named in the proposal  
2. Food waste forecasting  
3. Synthetic campus bin-level data generation  
4. Optional image-classification support  
5. Contamination risk modeling  
6. Policy simulation and optimization  

### Notes
- Some datasets require setup before download:
  - **Kaggle API** for Kaggle datasets
  - **Visual Crossing API key** for weather
  - Optional **Roboflow credentials**
- Public datasets may change schema over time, so the loaders are written to be schema-tolerant.
- The synthetic-campus layer is used to unify real and public data into a campus deployment scenario.

## 1. Optional: Mount Google Drive
Uncomment this cell in Colab if you want to save outputs to Drive.

In [26]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install dependencies
Run this once in Google Colab.

In [27]:

!pip -q install \
  "protobuf>=4.25.3,<5" \
  kaggle \
  xgboost \
  lightgbm \
  prophet \
  scikit-learn \
  statsmodels \
  ortools \
  roboflow \
  kagglehub

In [28]:
import tensorflow as tf
import xgboost
import lightgbm
import prophet
import sklearn
import statsmodels

print("All core imports worked")
print("TensorFlow:", tf.__version__)

All core imports worked
TensorFlow: 2.19.0


## 3. Imports and configuration

In [29]:

import os
import io
import re
import json
import math
import glob
import zipfile
import random
import warnings
import requests
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    f1_score,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception:
    PROPHET_AVAILABLE = False

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    TF_AVAILABLE = True
except Exception:
    TF_AVAILABLE = False

try:
    from ortools.linear_solver import pywraplp
    ORTOOLS_AVAILABLE = True
except Exception:
    ORTOOLS_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class Config:
    base_dir: str = "/content/campus_waste_intelligence"
    raw_dir: str = "/content/campus_waste_intelligence/raw"
    processed_dir: str = "/content/campus_waste_intelligence/processed"
    outputs_dir: str = "/content/campus_waste_intelligence/outputs"
    figures_dir: str = "/content/campus_waste_intelligence/figures"
    models_dir: str = "/content/campus_waste_intelligence/models"

    kaggle_food_waste: str = "joebeachcapital/food-waste"
    kaggle_waste_classification: str = "techsash/waste-classification-data"
    fikwaste_url: str = "https://osf.io/download/tyaj6/"
    trashnet_zip_url: str = "https://github.com/garythung/trashnet/archive/refs/heads/master.zip"
    realwaste_kaggle: str = "joebeachcapital/realwaste"

    visual_crossing_api_key: Optional[str] = None
    weather_location: str = "Birmingham,AL"
    weather_start_date: str = "2024-01-01"
    weather_end_date: str = "2024-12-31"

    roboflow_api_key: Optional[str] = None
    roboflow_workspace: Optional[str] = None
    roboflow_project: Optional[str] = None
    roboflow_version: Optional[int] = None

    campus_locations: Tuple[str, ...] = ("dining_hall", "dorm", "academic", "event_venue")
    start_date: str = "2024-01-01"
    end_date: str = "2024-12-31"
    use_hourly: bool = False

    forecast_target_name: str = "waste_volume"
    contamination_threshold: float = 0.25

CFG = Config()

for p in [CFG.base_dir, CFG.raw_dir, CFG.processed_dir, CFG.outputs_dir, CFG.figures_dir, CFG.models_dir]:
    os.makedirs(p, exist_ok=True)

print("Directories ready.")

Directories ready.


## 4. Helper utilities

In [30]:

def print_header(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def safe_read_csv(path: str, **kwargs) -> pd.DataFrame:
    encodings = ["utf-8", "latin1", "cp1252"]
    last_exc = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_exc = e
    raise last_exc

def find_files(root: str, patterns: List[str]) -> List[str]:
    files = []
    for pattern in patterns:
        files.extend(glob.glob(os.path.join(root, "**", pattern), recursive=True))
    return sorted(list(set(files)))

def normalize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_") for c in df.columns]
    return df

def infer_datetime_column(df: pd.DataFrame) -> Optional[str]:
    candidates = [c for c in df.columns if any(k in c for k in ["date", "time", "timestamp", "day"])]
    for c in candidates:
        try:
            parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().mean() > 0.5:
                return c
        except Exception:
            continue
    return None

def infer_numeric_target(df: pd.DataFrame) -> Optional[str]:
    candidates = []
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            score = 0
            if any(k in c for k in ["waste", "kg", "lb", "amount", "volume", "weight"]):
                score += 3
            if df[c].notna().mean() > 0.5:
                score += 1
            candidates.append((c, score))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def make_time_features(df: pd.DataFrame, dt_col: str) -> pd.DataFrame:
    df = df.copy()
    df[dt_col] = pd.to_datetime(df[dt_col])
    df["day_of_week"] = df[dt_col].dt.dayofweek
    df["week_of_year"] = df[dt_col].dt.isocalendar().week.astype(int)
    df["month"] = df[dt_col].dt.month
    df["quarter"] = df[dt_col].dt.quarter
    df["day_of_month"] = df[dt_col].dt.day
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["semester_week_proxy"] = ((df[dt_col].dt.dayofyear - 1) // 7 + 1).astype(int)
    return df

def add_lag_features(df: pd.DataFrame, group_cols: List[str], target_col: str, lags: List[int]) -> pd.DataFrame:
    df = df.copy().sort_values(group_cols + ["date"])

    # Standard lag features
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df.groupby(group_cols)[target_col].shift(lag)

    # Rolling mean features
    for window in [3, 7, 14]:
        df[f"{target_col}_rollmean_{window}"] = (
            df.groupby(group_cols)[target_col]
              .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
        )

    return df

def plot_forecast(actual: pd.Series, pred: pd.Series, title: str, path: str):
    plt.figure(figsize=(12, 5))
    plt.plot(actual.values, label="Actual")
    plt.plot(pred.values, label="Predicted")
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Waste Volume")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

def ensure_kaggle_credentials():
    kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(kaggle_path):
        print("[WARN] Kaggle API credentials not found at ~/.kaggle/kaggle.json")
        print("       Upload kaggle.json before running Kaggle download steps.")
        return False
    return True

## 5. Dataset download functions
Run this section if you want the notebook to download the proposal datasets automatically.

In [31]:

def download_kaggle_dataset(dataset_id: str, dest_dir: str):
    print_header(f"Downloading Kaggle dataset: {dataset_id}")
    os.makedirs(dest_dir, exist_ok=True)
    if not ensure_kaggle_credentials():
        print("Skipping Kaggle download.")
        return
    cmd = f"kaggle datasets download -d {dataset_id} -p {dest_dir} --unzip"
    rc = os.system(cmd)
    if rc != 0:
        print(f"[WARN] Kaggle download failed for {dataset_id}")
    else:
        print(f"Downloaded to: {dest_dir}")

def download_file(url: str, out_path: str):
    print(f"Downloading: {url}")
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

def download_and_extract_zip(url: str, dest_dir: str, zip_name: str = "archive.zip"):
    os.makedirs(dest_dir, exist_ok=True)
    zip_path = os.path.join(dest_dir, zip_name)
    download_file(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)
    print(f"Extracted to {dest_dir}")

def download_fikwaste(dest_dir: str):
    print_header("Downloading FIKWaste / external waste time-series source")
    os.makedirs(dest_dir, exist_ok=True)
    out_path = os.path.join(dest_dir, "fikwaste_download")
    try:
        download_file(CFG.fikwaste_url, out_path)
        print(f"Saved FIKWaste file to {out_path}")
    except Exception as e:
        print(f"[WARN] FIKWaste direct download failed: {e}")
        print("You may need to download it manually from the linked repository or source page.")

def download_trashnet(dest_dir: str):
    print_header("Downloading TrashNet")
    try:
        download_and_extract_zip(CFG.trashnet_zip_url, dest_dir, zip_name="trashnet_master.zip")
    except Exception as e:
        print(f"[WARN] TrashNet download failed: {e}")

def download_weather_visual_crossing(location: str, start_date: str, end_date: str, dest_csv: str):
    print_header("Downloading Visual Crossing weather data")
    api_key = CFG.visual_crossing_api_key or os.environ.get("VISUAL_CROSSING_API_KEY")
    if not api_key:
        print("[WARN] No Visual Crossing API key found. Skipping weather download.")
        return None

    url = (
        f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"
        f"{location}/{start_date}/{end_date}?unitGroup=metric&include=days&key={api_key}&contentType=json"
    )
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        days = pd.DataFrame(data.get("days", []))
        if len(days) == 0:
            print("[WARN] Weather API returned no daily records.")
            return None
        days.to_csv(dest_csv, index=False)
        print(f"Saved weather CSV to {dest_csv}")
        return days
    except Exception as e:
        print(f"[WARN] Weather download failed: {e}")
        return None

def try_download_roboflow(dest_dir: str):
    print_header("Downloading Roboflow dataset (optional)")
    if not CFG.roboflow_api_key or not CFG.roboflow_workspace or not CFG.roboflow_project or not CFG.roboflow_version:
        print("[INFO] Roboflow config not fully provided; skipping optional Roboflow download.")
        return None
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=CFG.roboflow_api_key)
        project = rf.workspace(CFG.roboflow_workspace).project(CFG.roboflow_project)
        dataset = project.version(CFG.roboflow_version).download("folder", location=dest_dir)
        print(f"Downloaded Roboflow dataset to {dest_dir}")
        return dataset
    except Exception as e:
        print(f"[WARN] Roboflow download failed: {e}")
        return None

def download_all_sources():
    download_kaggle_dataset(CFG.kaggle_food_waste, os.path.join(CFG.raw_dir, "food_waste_kaggle"))
    download_kaggle_dataset(CFG.kaggle_waste_classification, os.path.join(CFG.raw_dir, "waste_classification_kaggle"))
    download_kaggle_dataset(CFG.realwaste_kaggle, os.path.join(CFG.raw_dir, "realwaste_optional"))
    download_fikwaste(os.path.join(CFG.raw_dir, "fikwaste"))
    download_trashnet(os.path.join(CFG.raw_dir, "trashnet"))
    download_weather_visual_crossing(
        CFG.weather_location,
        CFG.weather_start_date,
        CFG.weather_end_date,
        os.path.join(CFG.raw_dir, "weather_visual_crossing.csv"),
    )
    try_download_roboflow(os.path.join(CFG.raw_dir, "roboflow_optional"))

### Optional: upload Kaggle credentials in Colab first

In [32]:
import json

kaggle_token = {
    "username": "SarthakTyagi99",
    "key": "KGAT_33e45332e7efbc6a3b9381ef8ebb00de"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_token, f)

print("kaggle.json created successfully")

kaggle.json created successfully


In [33]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [34]:
!kaggle datasets list

ref                                                            title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
nalisha/job-salary-prediction-dataset                          Job Salary Prediction Dataset                          3144815  2026-03-16 19:54:33.843000           6533        152                1  
ssssws/chocolate-sales-dataset-2023-2024                       Chocolate Sales Dataset 2023 - 2024                   24420255  2026-03-07 04:58:02.387000          10298        162                1  
asifxzaman/e-commerce-behavior-dataset8000-users               E-Commerce Behavior Dataset(8000 Users)                 151485  2026-03-31 16:43:13.683000            660         25                1  
grand

### Run downloads

In [35]:
download_kaggle_dataset(
    "techsash/waste-classification-data",
    os.path.join(CFG.raw_dir, "waste_classification_kaggle")
)

download_kaggle_dataset(
    "joebeachcapital/food-waste",
    os.path.join(CFG.raw_dir, "food_waste_kaggle")
)


Downloaded to: /content/campus_waste_intelligence/raw/waste_classification_kaggle

Downloaded to: /content/campus_waste_intelligence/raw/food_waste_kaggle


## 6. Food waste ingestion and standardization

In [36]:

def load_food_waste_sources(raw_root: str) -> Dict[str, pd.DataFrame]:
    print_header("Loading food waste sources")
    csv_files = find_files(raw_root, ["*.csv", "*.tsv", "*.txt", "*.xlsx", "*.xls"])
    frames = {}

    for fp in csv_files:
        name = Path(fp).stem.lower()
        try:
            if fp.endswith(".csv"):
                df = safe_read_csv(fp)
            elif fp.endswith(".tsv") or fp.endswith(".txt"):
                df = safe_read_csv(fp, sep=None, engine="python")
            elif fp.endswith(".xlsx") or fp.endswith(".xls"):
                df = pd.read_excel(fp)
            else:
                continue

            df = normalize_colnames(df)
            if len(df) > 0:
                frames[name] = df
                print(f"Loaded: {fp} -> shape={df.shape}")
        except Exception as e:
            print(f"[WARN] Could not load {fp}: {e}")

    return frames

def standardize_food_waste_df(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = normalize_colnames(df)
    dt_col = infer_datetime_column(df)
    target_col = infer_numeric_target(df)

    if dt_col is None:
        df = df.copy()
        df["date"] = pd.date_range(start=CFG.start_date, periods=len(df), freq="D")
        dt_col = "date"
    else:
        df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
        df = df[df[dt_col].notna()].copy()
        df = df.rename(columns={dt_col: "date"})

    if target_col is None:
        raise ValueError(f"No numeric waste-like target found in source={source_name}")

    possible_loc_cols = [c for c in df.columns if any(k in c for k in ["location", "site", "kitchen", "canteen", "restaurant", "facility"])]
    if possible_loc_cols:
        loc_col = possible_loc_cols[0]
        df["location_id"] = df[loc_col].astype(str).fillna(source_name)
    else:
        df["location_id"] = source_name

    possible_type_cols = [c for c in df.columns if any(k in c for k in ["waste_type", "category", "bin", "material", "class"])]
    if possible_type_cols:
        type_col = possible_type_cols[0]
        df["waste_type"] = df[type_col].astype(str).fillna("food")
    else:
        df["waste_type"] = "food"

    out = df[["date", "location_id", target_col, "waste_type"]].copy()
    out = out.rename(columns={target_col: "waste_volume"})
    out["source"] = source_name

    out["date"] = pd.to_datetime(out["date"]).dt.date
    out = out.groupby(["date", "location_id", "waste_type", "source"], as_index=False)["waste_volume"].sum()
    out["date"] = pd.to_datetime(out["date"])
    return out

def build_food_waste_master_table(raw_root: str) -> pd.DataFrame:
    frames = load_food_waste_sources(raw_root)
    standardized = []
    for name, df in frames.items():
        try:
            std = standardize_food_waste_df(df, name)
            standardized.append(std)
        except Exception as e:
            print(f"[WARN] Could not standardize {name}: {e}")

    if not standardized:
        raise RuntimeError("No food waste sources could be standardized. Check raw files.")

    master = pd.concat(standardized, ignore_index=True)
    master = master.sort_values(["location_id", "date"]).reset_index(drop=True)
    master.to_csv(os.path.join(CFG.processed_dir, "food_waste_master.csv"), index=False)
    print(f"Saved master food waste table with shape {master.shape}")
    return master

In [37]:
master_df = build_food_waste_master_table(CFG.raw_dir)


Loading food waste sources
Loaded: /content/campus_waste_intelligence/raw/food_waste_kaggle/Food Waste data and research - by country.csv -> shape=(214, 12)
Saved master food waste table with shape (214, 5)


## 7. Context enrichment: weather, event flags, foot traffic

In [38]:

def load_weather_if_available() -> Optional[pd.DataFrame]:
    fp = os.path.join(CFG.raw_dir, "weather_visual_crossing.csv")
    if os.path.exists(fp):
        df = safe_read_csv(fp)
        df = normalize_colnames(df)
        if "datetime" in df.columns:
            df = df.rename(columns={"datetime": "date"})
        elif "date" not in df.columns:
            dt_col = infer_datetime_column(df)
            if dt_col:
                df = df.rename(columns={dt_col: "date"})
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"]).dt.normalize()
            return df
    return None

def make_synthetic_event_calendar(start_date: str, end_date: str) -> pd.DataFrame:
    dates = pd.date_range(start_date, end_date, freq="D")
    df = pd.DataFrame({"date": dates})
    df["day_of_week"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month

    df["event_flag"] = 0
    event_prob = np.where(df["day_of_week"].isin([4, 5]), 0.30, 0.08)
    event_prob += np.where(df["month"].isin([9, 10, 11]), 0.08, 0.00)
    event_prob += np.where(df["month"].isin([3, 4]), 0.05, 0.00)
    rng = np.random.default_rng(SEED)
    df["event_flag"] = (rng.random(len(df)) < event_prob).astype(int)
    df["event_intensity"] = np.where(df["event_flag"] == 1, rng.integers(1, 4, size=len(df)), 0)
    return df[["date", "event_flag", "event_intensity"]]

def enrich_with_context(food_df: pd.DataFrame) -> pd.DataFrame:
    df = food_df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df = make_time_features(df, "date")

    weather = load_weather_if_available()
    if weather is not None:
        weather_cols = [c for c in weather.columns if c != "date"]
        df = df.merge(weather[["date"] + weather_cols], on="date", how="left")
    else:
        rng = np.random.default_rng(SEED)
        df["temp"] = 18 + 10 * np.sin(2 * np.pi * df["month"] / 12) + rng.normal(0, 2, len(df))
        df["precip"] = np.clip(rng.gamma(shape=1.2, scale=1.5, size=len(df)) - 1.0, 0, None)

    events = make_synthetic_event_calendar(df["date"].min().date().isoformat(), df["date"].max().date().isoformat())
    df = df.merge(events, on="date", how="left")

    base_by_location = {
        "dining_hall": 1.6,
        "dorm": 1.2,
        "academic": 1.0,
        "event_venue": 1.8,
    }
    df["location_type"] = df["location_id"].astype(str).apply(
        lambda x: "dining_hall" if "dining" in x.lower() or "canteen" in x.lower() else (
            "dorm" if "dorm" in x.lower() else (
                "event_venue" if "event" in x.lower() or "stadium" in x.lower() else "academic"
            )
        )
    )
    rng = np.random.default_rng(SEED)
    df["foot_traffic_proxy"] = (
        df["location_type"].map(base_by_location).fillna(1.0)
        * (1.0 + 0.35 * df["event_flag"] + 0.12 * df["is_weekend"])
        * (1.0 + rng.normal(0, 0.05, len(df)))
        * 100
    )
    return df

## 8. Forecasting pipeline
This section corresponds to Dennis's part.

In [39]:

def prepare_forecasting_table(master_df: pd.DataFrame) -> pd.DataFrame:
    df = enrich_with_context(master_df)

    daily = (
        df.groupby(["date", "location_id", "location_type"], as_index=False)
          .agg({
              "waste_volume": "sum",
              "event_flag": "max",
              "event_intensity": "max",
              "foot_traffic_proxy": "mean",
              "temp": "mean" if "temp" in df.columns else "size",
              "precip": "mean" if "precip" in df.columns else "size",
              "day_of_week": "first",
              "week_of_year": "first",
              "month": "first",
              "is_weekend": "first",
              "semester_week_proxy": "first",
          })
    )

    if "temp" not in daily.columns:
        daily["temp"] = np.nan
    if "precip" not in daily.columns:
        daily["precip"] = np.nan

    daily = add_lag_features(daily.rename(columns={"date": "date"}), ["location_id"], "waste_volume", [1, 2, 3, 7, 14])
    daily = daily.dropna().reset_index(drop=True)
    return daily

def temporal_train_val_test_split(df: pd.DataFrame, date_col: str = "date", train_frac: float = 0.7, val_frac: float = 0.15):
    df = df.sort_values(date_col).reset_index(drop=True)
    unique_dates = sorted(df[date_col].unique())
    n = len(unique_dates)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train_dates = unique_dates[:train_end]
    val_dates = unique_dates[train_end:val_end]
    test_dates = unique_dates[val_end:]

    train_df = df[df[date_col].isin(train_dates)].copy()
    val_df = df[df[date_col].isin(val_dates)].copy()
    test_df = df[df[date_col].isin(test_dates)].copy()
    return train_df, val_df, test_df

def fit_xgb_forecaster(train_df: pd.DataFrame, val_df: pd.DataFrame, feature_cols: List[str], target_col: str = "waste_volume"):
    model = XGBRegressor(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        objective="reg:squarederror",
    )
    model.fit(train_df[feature_cols], train_df[target_col])
    val_pred = model.predict(val_df[feature_cols])
    return model, val_pred

def fit_lgbm_forecaster(train_df: pd.DataFrame, val_df: pd.DataFrame, feature_cols: List[str], target_col: str = "waste_volume"):
    model = LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        random_state=SEED,
    )
    model.fit(train_df[feature_cols], train_df[target_col])
    val_pred = model.predict(val_df[feature_cols])
    return model, val_pred

def run_forecasting_pipeline(master_df: pd.DataFrame) -> pd.DataFrame:
    print_header("Running forecasting pipeline")
    daily = prepare_forecasting_table(master_df)

    feature_cols = [
        c for c in daily.columns
        if c not in ["date", "location_id", "location_type", "waste_volume"]
        and pd.api.types.is_numeric_dtype(daily[c])
    ]

    train_df, val_df, test_df = temporal_train_val_test_split(daily)

    xgb_model, xgb_val_pred = fit_xgb_forecaster(train_df, val_df, feature_cols)
    lgbm_model, lgbm_val_pred = fit_lgbm_forecaster(train_df, val_df, feature_cols)

    xgb_val_rmse = rmse(val_df["waste_volume"], xgb_val_pred)
    lgbm_val_rmse = rmse(val_df["waste_volume"], lgbm_val_pred)

    best_model_name = "xgboost" if xgb_val_rmse <= lgbm_val_rmse else "lightgbm"
    best_model = xgb_model if best_model_name == "xgboost" else lgbm_model

    print(f"Validation RMSE - XGBoost: {xgb_val_rmse:.4f}")
    print(f"Validation RMSE - LightGBM: {lgbm_val_rmse:.4f}")
    print(f"Selected best model: {best_model_name}")

    test_pred = best_model.predict(test_df[feature_cols])

    results = test_df[["date", "location_id", "waste_volume"]].copy()
    results["y_true"] = results["waste_volume"]
    results["y_pred"] = test_pred
    results["y_pred_lower"] = np.clip(results["y_pred"] - results["y_pred"].std(), 0, None)
    results["y_pred_upper"] = results["y_pred"] + results["y_pred"].std()
    results = results.drop(columns=["waste_volume"])

    out_path = os.path.join(CFG.outputs_dir, "food_waste_forecast.csv")
    results.to_csv(out_path, index=False)

    metrics = pd.DataFrame([{
        "model": best_model_name,
        "mae": mean_absolute_error(results["y_true"], results["y_pred"]),
        "rmse": rmse(results["y_true"], results["y_pred"]),
    }])
    metrics.to_csv(os.path.join(CFG.outputs_dir, "forecast_metrics.csv"), index=False)

    plot_forecast(
        results["y_true"].reset_index(drop=True),
        results["y_pred"].reset_index(drop=True),
        title=f"Food Waste Forecast ({best_model_name})",
        path=os.path.join(CFG.figures_dir, "forecast_vs_actual.png"),
    )

    print(f"Saved forecast outputs to {out_path}")
    return results

### Run the forecasting stage

In [40]:

master_df = build_food_waste_master_table(CFG.raw_dir)
forecast_df = run_forecasting_pipeline(master_df)

display(master_df.head())
display(forecast_df.head())


Loading food waste sources
Loaded: /content/campus_waste_intelligence/raw/food_waste_kaggle/Food Waste data and research - by country.csv -> shape=(214, 12)
Saved master food waste table with shape (214, 5)

Running forecasting pipeline
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000102 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 447
[LightGBM] [Info] Number of data points in the train set: 140, number of used features: 18
[LightGBM] [Info] Start training from score 127.185714
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

,date,location_id,waste_type,source,waste_volume
0,2024-01-01,food waste data and research - by country,126,food waste data and research - by country,126
1,2024-01-02,food waste data and research - by country,127,food waste data and research - by country,127
2,2024-01-03,food waste data and research - by country,135,food waste data and research - by country,135
3,2024-01-04,food waste data and research - by country,123,food waste data and research - by country,123
4,2024-01-05,food waste data and research - by country,144,food waste data and research - by country,144


,date,location_id,y_true,y_pred,y_pred_lower,y_pred_upper
170,2024-07-03,food waste data and research - by country,116,112.397224,106.759109,118.035339
171,2024-07-04,food waste data and research - by country,112,119.007683,113.369568,124.645798
172,2024-07-05,food waste data and research - by country,125,117.738235,112.100128,123.376343
173,2024-07-06,food waste data and research - by country,148,113.593651,107.955536,119.231766
174,2024-07-07,food waste data and research - by country,141,127.768089,122.129974,133.406204


## 9. Optional image dataset support
These helpers collect image files from the waste image datasets.  
The training function is optional and only runs if TensorFlow is available and enough images are present.

In [41]:

def gather_image_paths(root_dir: str) -> pd.DataFrame:
    patterns = ["*.jpg", "*.jpeg", "*.png", "*.webp"]
    img_files = find_files(root_dir, patterns)
    rows = []
    for fp in img_files:
        cls = Path(fp).parent.name
        rows.append({"filepath": fp, "label": cls})
    return pd.DataFrame(rows)

def normalize_waste_image_labels(label: str) -> str:
    label = str(label).lower()
    if any(k in label for k in ["organic", "food", "bio"]):
        return "organic"
    if any(k in label for k in ["recycle", "recyclable", "paper", "plastic", "metal", "glass", "cardboard"]):
        return "recyclable"
    return "landfill"

def build_image_manifest() -> pd.DataFrame:
    root_dirs = [
        os.path.join(CFG.raw_dir, "waste_classification_kaggle"),
        os.path.join(CFG.raw_dir, "trashnet"),
        os.path.join(CFG.raw_dir, "realwaste_optional"),
        os.path.join(CFG.raw_dir, "roboflow_optional"),
    ]
    manifests = []
    for rd in root_dirs:
        if os.path.exists(rd):
            df = gather_image_paths(rd)
            if len(df) > 0:
                df["label_standard"] = df["label"].apply(normalize_waste_image_labels)
                manifests.append(df)
    if not manifests:
        print("[INFO] No image datasets found. Skipping CV module.")
        return pd.DataFrame(columns=["filepath", "label", "label_standard"])
    manifest = pd.concat(manifests, ignore_index=True).drop_duplicates()
    manifest.to_csv(os.path.join(CFG.processed_dir, "waste_image_manifest.csv"), index=False)
    return manifest

In [42]:

def train_waste_image_classifier(manifest: pd.DataFrame, epochs: int = 3, batch_size: int = 32):
    if not TF_AVAILABLE:
        print("[WARN] TensorFlow unavailable. Skipping image classifier.")
        return None
    if len(manifest) < 100:
        print("[WARN] Not enough image samples to train a robust classifier. Skipping.")
        return None

    print_header("Training image-based waste classifier")
    manifest = manifest.copy()
    manifest = manifest[manifest["label_standard"].isin(["organic", "recyclable", "landfill"])].reset_index(drop=True)

    train_df, val_df = train_test_split(
        manifest,
        test_size=0.2,
        random_state=SEED,
        stratify=manifest["label_standard"],
    )

    class_names = sorted(train_df["label_standard"].unique())
    label_to_idx = {label: i for i, label in enumerate(class_names)}

    def make_dataset(df_subset, shuffle=True):
        paths = df_subset["filepath"].tolist()
        labels = [label_to_idx[x] for x in df_subset["label_standard"].tolist()]
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))

        def load_img(path, label):
            img = tf.io.read_file(path)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, (224, 224))
            img = tf.cast(img, tf.float32)
            label = tf.one_hot(label, depth=len(class_names))
            return img, label

        ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
        if shuffle:
            ds = ds.shuffle(buffer_size=min(1000, len(df_subset)), seed=SEED)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(train_df, shuffle=True)
    val_ds = make_dataset(val_df, shuffle=False)

    base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)
    model = models.Model(inputs, outputs)

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=epochs)

    model_path = os.path.join(CFG.models_dir, "waste_image_classifier.keras")
    model.save(model_path)
    print(f"Saved image classifier to {model_path}")
    return model

In [43]:

manifest = build_image_manifest()
display(manifest.head())

# Optional:
# image_model = train_waste_image_classifier(manifest, epochs=3, batch_size=32)

,filepath,label,label_standard
0,/content/campus_waste_intelligence/raw/waste_c...,O,landfill
1,/content/campus_waste_intelligence/raw/waste_c...,O,landfill
2,/content/campus_waste_intelligence/raw/waste_c...,O,landfill
3,/content/campus_waste_intelligence/raw/waste_c...,O,landfill
4,/content/campus_waste_intelligence/raw/waste_c...,O,landfill


## 10. Synthetic campus bin-level data + contamination model
This section corresponds to Nicholas's part.

In [44]:

def create_synthetic_campus_table(forecast_df: pd.DataFrame) -> pd.DataFrame:
    print_header("Creating synthetic campus bin-level data")
    forecast_df = forecast_df.copy()
    forecast_df["date"] = pd.to_datetime(forecast_df["date"]).dt.normalize()

    context = make_synthetic_event_calendar(forecast_df["date"].min().date().isoformat(), forecast_df["date"].max().date().isoformat())
    df = forecast_df.merge(context, on="date", how="left")

    df["location_type"] = df["location_id"].astype(str).apply(
        lambda x: "dining" if "dining" in x.lower() or "canteen" in x.lower() else (
            "dorm" if "dorm" in x.lower() else (
                "event" if "event" in x.lower() or "stadium" in x.lower() else "academic"
            )
        )
    )

    rows = []
    rng = np.random.default_rng(SEED)
    for _, row in df.iterrows():
        num_bins = 3 if row["location_type"] == "dining" else 2
        for bin_idx in range(num_bins):
            bin_type = ["food", "recycle", "landfill"][bin_idx % 3]
            bin_capacity = {
                "food": rng.integers(50, 90),
                "recycle": rng.integers(40, 80),
                "landfill": rng.integers(45, 85),
            }[bin_type]
            pickup_frequency = rng.choice([1, 2, 3])
            alt_bin_distance = rng.uniform(2, 15)
            signage_strength = rng.choice([0, 1, 2])
            crowdedness = (1 + 0.4 * row["event_flag"] + rng.normal(0, 0.1)) * (1.2 if row["location_type"] == "dining" else 1.0)
            waste_share = {"food": 0.55, "recycle": 0.25, "landfill": 0.20}[bin_type]
            expected_fill = max(0, row["y_pred"] * waste_share / max(bin_capacity, 1))
            overflow_risk = max(0, expected_fill - 1.0)

            p_contam = (
                0.08
                + 0.12 * overflow_risk
                + 0.10 * row["event_flag"]
                + 0.08 * (row["location_type"] == "event")
                + 0.06 * (row["location_type"] == "dorm")
                + 0.05 * (alt_bin_distance / 15)
                + 0.08 * (pickup_frequency == 1)
                - 0.05 * signage_strength
                + rng.normal(0, 0.03)
            )
            p_contam = float(np.clip(p_contam, 0.01, 0.95))
            contam_true = int(rng.random() < p_contam)
            contam_pct_true = float(np.clip(p_contam + rng.normal(0, 0.05), 0.0, 1.0))

            rows.append({
                "timestamp": row["date"],
                "location_id": row["location_id"],
                "bin_id": f"{row['location_id']}_bin_{bin_idx+1}",
                "bin_type": bin_type,
                "location_type": row["location_type"],
                "predicted_waste_volume": row["y_pred"],
                "event_flag": row["event_flag"],
                "event_intensity": row["event_intensity"],
                "bin_capacity": bin_capacity,
                "pickup_frequency": pickup_frequency,
                "alt_bin_distance": alt_bin_distance,
                "signage_strength": signage_strength,
                "crowdedness": crowdedness,
                "expected_fill": expected_fill,
                "overflow_risk": overflow_risk,
                "p_contam_true_latent": p_contam,
                "contam_true": contam_true,
                "contam_pct_true": contam_pct_true,
            })

    campus = pd.DataFrame(rows)
    campus.to_csv(os.path.join(CFG.processed_dir, "synthetic_campus_bins.csv"), index=False)
    return campus

def train_contamination_model(campus_df: pd.DataFrame) -> pd.DataFrame:
    print_header("Training contamination risk model")
    df = campus_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month

    feature_cols = [
        "bin_type",
        "location_type",
        "predicted_waste_volume",
        "event_flag",
        "event_intensity",
        "bin_capacity",
        "pickup_frequency",
        "alt_bin_distance",
        "signage_strength",
        "crowdedness",
        "expected_fill",
        "overflow_risk",
        "day_of_week",
        "month",
    ]
    target_col = "contam_true"

    dates = sorted(df["timestamp"].dt.date.unique())
    n = len(dates)
    train_cut = int(n * 0.7)
    val_cut = int(n * 0.85)

    train_dates = dates[:train_cut]
    val_dates = dates[train_cut:val_cut]
    test_dates = dates[val_cut:]

    train_df = df[df["timestamp"].dt.date.isin(train_dates)].copy()
    val_df = df[df["timestamp"].dt.date.isin(val_dates)].copy()
    test_df = df[df["timestamp"].dt.date.isin(test_dates)].copy()

    cat_cols = ["bin_type", "location_type"]
    all_df = pd.concat([train_df[feature_cols], val_df[feature_cols], test_df[feature_cols]], axis=0).copy()
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        all_df[col] = le.fit_transform(all_df[col].astype(str))
        encoders[col] = le

    transformed_train = train_df[feature_cols].copy()
    transformed_val = val_df[feature_cols].copy()
    transformed_test = test_df[feature_cols].copy()

    for col, le in encoders.items():
        transformed_train[col] = le.transform(transformed_train[col].astype(str))
        transformed_val[col] = le.transform(transformed_val[col].astype(str))
        transformed_test[col] = le.transform(transformed_test[col].astype(str))

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        eval_metric="logloss",
    )
    xgb.fit(transformed_train[feature_cols], train_df[target_col])

    calibrator = CalibratedClassifierCV(xgb, method="isotonic", cv="prefit")
    calibrator.fit(transformed_val[feature_cols], val_df[target_col])

    p_test = calibrator.predict_proba(transformed_test[feature_cols])[:, 1]
    y_test_pred = (p_test >= 0.5).astype(int)

    metrics = {
        "auc": roc_auc_score(test_df[target_col], p_test),
        "f1": f1_score(test_df[target_col], y_test_pred),
        "brier": brier_score_loss(test_df[target_col], p_test),
    }
    print("Contamination model metrics:", metrics)

    results = test_df[["timestamp", "bin_id", "location_id", "contam_true"]].copy()
    results["p_contam"] = p_test
    results["contam_pred"] = y_test_pred

    out_path = os.path.join(CFG.outputs_dir, "contamination_predictions.csv")
    results.to_csv(out_path, index=False)

    pd.DataFrame([metrics]).to_csv(os.path.join(CFG.outputs_dir, "contamination_metrics.csv"), index=False)

    prob_true, prob_pred = calibration_curve(test_df[target_col], p_test, n_bins=10)
    plt.figure(figsize=(6, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Contamination Model Reliability Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.figures_dir, "contamination_reliability_curve.png"))
    plt.close()

    return results

In [45]:

campus_df = create_synthetic_campus_table(forecast_df)
contamination_preds = train_contamination_model(campus_df)

display(campus_df.head())
display(contamination_preds.head())


Creating synthetic campus bin-level data

Training contamination risk model
Contamination model metrics: {'auc': np.float64(0.6666666666666667), 'f1': 0.0, 'brier': np.float64(0.09044051305964176)}


,timestamp,location_id,bin_id,bin_type,location_type,predicted_waste_volume,event_flag,event_intensity,bin_capacity,pickup_frequency,alt_bin_distance,signage_strength,crowdedness,expected_fill,overflow_risk,p_contam_true_latent,contam_true,contam_pct_true
0,2024-07-03,food waste data and research - by country,food waste data and research - by country_bin_1,food,academic,112.397224,0,0,53,2,13.161773,0,0.804896,1.166386,0.166386,0.104774,0,0.088961
1,2024-07-03,food waste data and research - by country,food waste data and research - by country_bin_2,recycle,academic,112.397224,0,0,60,3,6.820374,1,1.077779,0.468322,0.000000,0.054716,0,0.078091
2,2024-07-04,food waste data and research - by country,food waste data and research - by country_bin_1,food,academic,119.007683,0,0,68,2,2.829624,2,0.995007,0.962562,0.000000,0.010000,0,0.071127
3,2024-07-04,food waste data and research - by country,food waste data and research - by country_bin_2,recycle,academic,119.007683,0,0,57,3,4.530303,2,1.053231,0.521964,0.000000,0.010000,0,0.031541
4,2024-07-05,food waste data and research - by country,food waste data and research - by country_bin_1,food,academic,117.738235,0,0,86,3,6.235730,2,1.061598,0.752977,0.000000,0.034655,0,0.000000


,timestamp,bin_id,location_id,contam_true,p_contam,contam_pred
50,2024-07-28,food waste data and research - by country_bin_1,food waste data and research - by country,0,0.166667,0
51,2024-07-28,food waste data and research - by country_bin_2,food waste data and research - by country,0,0.166667,0
52,2024-07-29,food waste data and research - by country_bin_1,food waste data and research - by country,0,0.141607,0
53,2024-07-29,food waste data and research - by country_bin_2,food waste data and research - by country,1,0.166667,0
54,2024-07-30,food waste data and research - by country_bin_1,food waste data and research - by country,0,0.166667,0


## 11. Policy / optimization layer
This section corresponds to Sarthak's part.

In [46]:

def merge_for_policy(campus_df: pd.DataFrame, contamination_preds: pd.DataFrame) -> pd.DataFrame:
    merged = campus_df.merge(
        contamination_preds[["timestamp", "bin_id", "p_contam", "contam_pred"]],
        on=["timestamp", "bin_id"],
        how="left",
    )
    merged["p_contam"] = merged["p_contam"].fillna(merged["p_contam_true_latent"])
    merged["contam_pred"] = merged["contam_pred"].fillna((merged["p_contam"] >= 0.5).astype(int))
    return merged

def compute_operational_costs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    hauling_cost = 12.0 / df["pickup_frequency"].clip(lower=1)
    contamination_cost = 60.0 * df["p_contam"]
    overflow_penalty = 80.0 * np.clip(df["expected_fill"] - 1.0, 0, None)
    carbon_proxy = 8.0 * df["predicted_waste_volume"] / 100.0

    df["cost_hauling"] = hauling_cost
    df["cost_contamination"] = contamination_cost
    df["cost_overflow"] = overflow_penalty
    df["cost_carbon_proxy"] = carbon_proxy
    df["total_cost"] = df[["cost_hauling", "cost_contamination", "cost_overflow", "cost_carbon_proxy"]].sum(axis=1)
    return df

def simulate_intervention(df: pd.DataFrame, intervention: str) -> pd.DataFrame:
    out = df.copy()

    if intervention == "baseline":
        return compute_operational_costs(out)

    if intervention == "increase_pickup_high_risk":
        mask = (out["p_contam"] > 0.45) | (out["expected_fill"] > 0.9)
        out.loc[mask, "pickup_frequency"] = np.minimum(out.loc[mask, "pickup_frequency"] + 1, 4)
        out.loc[mask, "expected_fill"] = out.loc[mask, "expected_fill"] * 0.85
        out.loc[mask, "p_contam"] = np.clip(out.loc[mask, "p_contam"] * 0.92, 0, 1)

    elif intervention == "improved_signage":
        mask = out["location_type"].isin(["dorm", "event", "academic"])
        out.loc[mask, "signage_strength"] = np.minimum(out.loc[mask, "signage_strength"] + 1, 2)
        out.loc[mask, "p_contam"] = np.clip(out.loc[mask, "p_contam"] * 0.85, 0, 1)

    elif intervention == "add_bins_high_traffic":
        mask = out["expected_fill"] > 0.95
        out.loc[mask, "bin_capacity"] = out.loc[mask, "bin_capacity"] * 1.25
        out.loc[mask, "expected_fill"] = out.loc[mask, "expected_fill"] / 1.25
        out.loc[mask, "p_contam"] = np.clip(out.loc[mask, "p_contam"] * 0.88, 0, 1)

    elif intervention == "paired_bins_strategy":
        mask = out["location_type"].isin(["academic", "event", "dorm"])
        out.loc[mask, "alt_bin_distance"] = out.loc[mask, "alt_bin_distance"] * 0.75
        out.loc[mask, "p_contam"] = np.clip(out.loc[mask, "p_contam"] * 0.83, 0, 1)

    elif intervention == "combined_strategy":
        out = simulate_intervention(out, "increase_pickup_high_risk")
        out = simulate_intervention(out, "improved_signage")
        out = simulate_intervention(out, "add_bins_high_traffic")
        out = simulate_intervention(out, "paired_bins_strategy")
        return compute_operational_costs(out)

    return compute_operational_costs(out)

def optimize_budgeted_actions(df: pd.DataFrame, budget: float = 1500.0) -> pd.DataFrame:
    if not ORTOOLS_AVAILABLE:
        print("[INFO] OR-Tools unavailable; falling back to greedy strategy.")
        return greedy_budgeted_actions(df, budget)

    print_header("Running budgeted optimization")
    temp = df.copy().reset_index(drop=True)
    temp["benefit"] = 65.0 * temp["p_contam"] + 50.0 * np.clip(temp["expected_fill"] - 0.9, 0, None)
    temp["action_cost"] = 45.0 + 15.0 * (temp["location_type"].isin(["event", "dorm"]).astype(int))

    solver = pywraplp.Solver.CreateSolver("SCIP")
    x = {i: solver.IntVar(0, 1, f"x_{i}") for i in temp.index}

    solver.Add(sum(x[i] * float(temp.loc[i, "action_cost"]) for i in temp.index) <= budget)
    solver.Maximize(sum(x[i] * float(temp.loc[i, "benefit"]) for i in temp.index))

    status = solver.Solve()
    if status not in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        print("[WARN] Optimization failed; using greedy fallback.")
        return greedy_budgeted_actions(df, budget)

    chosen = [i for i in temp.index if x[i].solution_value() > 0.5]
    temp["optimized_action"] = 0
    temp.loc[chosen, "optimized_action"] = 1

    mask = temp["optimized_action"] == 1
    temp.loc[mask, "pickup_frequency"] = np.minimum(temp.loc[mask, "pickup_frequency"] + 1, 4)
    temp.loc[mask, "signage_strength"] = np.minimum(temp.loc[mask, "signage_strength"] + 1, 2)
    temp.loc[mask, "bin_capacity"] = temp.loc[mask, "bin_capacity"] * 1.15
    temp.loc[mask, "expected_fill"] = temp.loc[mask, "expected_fill"] * 0.82
    temp.loc[mask, "p_contam"] = np.clip(temp.loc[mask, "p_contam"] * 0.80, 0, 1)

    return compute_operational_costs(temp)

def greedy_budgeted_actions(df: pd.DataFrame, budget: float = 1500.0) -> pd.DataFrame:
    temp = df.copy()
    temp["benefit_score"] = 65.0 * temp["p_contam"] + 50.0 * np.clip(temp["expected_fill"] - 0.9, 0, None)
    temp["action_cost"] = 45.0 + 15.0 * (temp["location_type"].isin(["event", "dorm"]).astype(int))
    temp = temp.sort_values("benefit_score", ascending=False).copy()
    temp["optimized_action"] = 0

    spent = 0.0
    chosen_idx = []
    for idx, row in temp.iterrows():
        c = float(row["action_cost"])
        if spent + c <= budget:
            spent += c
            chosen_idx.append(idx)
    temp.loc[chosen_idx, "optimized_action"] = 1

    mask = temp["optimized_action"] == 1
    temp.loc[mask, "pickup_frequency"] = np.minimum(temp.loc[mask, "pickup_frequency"] + 1, 4)
    temp.loc[mask, "signage_strength"] = np.minimum(temp.loc[mask, "signage_strength"] + 1, 2)
    temp.loc[mask, "bin_capacity"] = temp.loc[mask, "bin_capacity"] * 1.15
    temp.loc[mask, "expected_fill"] = temp.loc[mask, "expected_fill"] * 0.82
    temp.loc[mask, "p_contam"] = np.clip(temp.loc[mask, "p_contam"] * 0.80, 0, 1)

    return compute_operational_costs(temp)

def run_policy_layer(campus_df: pd.DataFrame, contamination_preds: pd.DataFrame):
    print_header("Running policy layer")
    merged = merge_for_policy(campus_df, contamination_preds)

    scenarios = {}
    for intervention in [
        "baseline",
        "increase_pickup_high_risk",
        "improved_signage",
        "add_bins_high_traffic",
        "paired_bins_strategy",
        "combined_strategy",
    ]:
        scenarios[intervention] = simulate_intervention(merged, intervention)

    scenarios["budget_optimized"] = optimize_budgeted_actions(merged, budget=1500.0)

    summary_rows = []
    baseline_cost = scenarios["baseline"]["total_cost"].sum()
    baseline_contam = scenarios["baseline"]["p_contam"].mean()

    for name, sdf in scenarios.items():
        total_cost = sdf["total_cost"].sum()
        mean_contam = sdf["p_contam"].mean()
        mean_fill = sdf["expected_fill"].mean()
        summary_rows.append({
            "scenario": name,
            "total_cost": total_cost,
            "cost_savings_vs_baseline": baseline_cost - total_cost,
            "mean_contamination": mean_contam,
            "contamination_reduction_vs_baseline": baseline_contam - mean_contam,
            "mean_expected_fill": mean_fill,
        })

    summary = pd.DataFrame(summary_rows).sort_values("total_cost")
    summary_path = os.path.join(CFG.outputs_dir, "policy_results.csv")
    summary.to_csv(summary_path, index=False)

    plt.figure(figsize=(10, 5))
    plt.bar(summary["scenario"], summary["cost_savings_vs_baseline"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Cost savings vs baseline")
    plt.title("Policy Scenario Cost Savings")
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.figures_dir, "policy_cost_savings.png"))
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(summary["scenario"], summary["contamination_reduction_vs_baseline"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Contamination reduction vs baseline")
    plt.title("Policy Scenario Impact on Contamination")
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.figures_dir, "policy_contamination_reduction.png"))
    plt.close()

    return summary

In [47]:

policy_summary = run_policy_layer(campus_df, contamination_preds)
display(policy_summary)


Running policy layer

Running budgeted optimization


,scenario,total_cost,cost_savings_vs_baseline,mean_contamination,contamination_reduction_vs_baseline,mean_expected_fill
6,budget_optimized,1161.044013,232.688726,0.084512,0.015990,0.626076
5,combined_strategy,1190.034062,203.698677,0.069258,0.031244,0.649712
1,increase_pickup_high_risk,1303.946134,89.786605,0.098927,0.001575,0.663244
3,add_bins_high_traffic,1324.521662,69.211078,0.098819,0.001683,0.664361
4,paired_bins_strategy,1332.225700,61.507039,0.083416,0.017085,0.696762
2,improved_signage,1339.461822,54.270917,0.085426,0.015075,0.696762
0,baseline,1393.732739,0.000000,0.100502,0.000000,0.696762


The policy simulation results show that all intervention strategies outperformed the baseline scenario in terms of cost and contamination reduction. The budget-optimized strategy achieved the lowest total operational cost (1161.04), producing a cost savings of 232.69 compared to baseline. However, the combined strategy yielded the strongest contamination reduction, lowering mean contamination from 0.1005 to 0.0693 while still reducing total cost to 1190.03. These findings suggest that optimization-based interventions are effective when minimizing cost is the primary objective, whereas combined operational measures are preferable when reducing contamination is the main priority.
Our results show that intervention design matters: a budget-aware optimization strategy minimizes operational cost most effectively, while a combined intervention strategy achieves the strongest reduction in contamination risk.

## 12. Full pipeline runner
Use this if you want to run everything in one shot.

In [48]:

def main():
    print_header("Campus Waste Intelligence System - Full Pipeline")

    # Optional: uncomment when you want auto-download
    # download_all_sources()

    master_df = build_food_waste_master_table(CFG.raw_dir)
    forecast_df = run_forecasting_pipeline(master_df)

    manifest = build_image_manifest()
    if len(manifest) > 0:
        print(f"Found {len(manifest)} images across datasets.")
        # Optional image model
        # _ = train_waste_image_classifier(manifest, epochs=3, batch_size=32)

    campus_df = create_synthetic_campus_table(forecast_df)
    contamination_preds = train_contamination_model(campus_df)
    policy_summary = run_policy_layer(campus_df, contamination_preds)

    print_header("Pipeline completed successfully")
    print("Outputs:", CFG.outputs_dir)
    print("Figures:", CFG.figures_dir)
    print("Processed data:", CFG.processed_dir)
    return {
        "master_df": master_df,
        "forecast_df": forecast_df,
        "campus_df": campus_df,
        "contamination_preds": contamination_preds,
        "policy_summary": policy_summary,
    }

# Example:
# results = main()

## 13. Output files produced
When the notebook runs successfully, it saves:

- `processed/food_waste_master.csv`
- `outputs/food_waste_forecast.csv`
- `outputs/forecast_metrics.csv`
- `processed/synthetic_campus_bins.csv`
- `outputs/contamination_predictions.csv`
- `outputs/contamination_metrics.csv`
- `outputs/policy_results.csv`

And figures such as:
- `figures/forecast_vs_actual.png`
- `figures/contamination_reliability_curve.png`
- `figures/policy_cost_savings.png`
- `figures/policy_contamination_reduction.png`